In [1]:
import tensorflow as tf
import numpy as np
import os
import time

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from sklearn.metrics import classification_report


In [2]:
DATASET_PATH = r"D:\Project Jupyter\MODELLING\Dataset_baruPR"
TEST_DIR = os.path.join(DATASET_PATH, "test")

PRUNED_MODEL_PATH = r"D:\mobilenetv3_se_attention_pruned_final20_30_40FINALAGP10"
TFLITE_FP16_PATH  = r"D:\Project Jupyter\mobilenetv3_FINAL_FP16FIXNEWBANGET.tflite" 

# PARAMETER
IMG_SIZE = (224, 224)
BATCH_SIZE = 1            

In [3]:
model_fp32 = tf.keras.models.load_model(
    PRUNED_MODEL_PATH,
    compile=False
)

print("Model pruned FP32 berhasil dimuat")


Model pruned FP32 berhasil dimuat


In [4]:
def preprocess_image(image):
    image = tf.image.resize_with_pad(image, IMG_SIZE[0], IMG_SIZE[1])
    image = preprocess_input(image)
    return image

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_image
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

class_names = list(test_generator.class_indices.keys())
print("Class:", class_names)


Found 232 images belonging to 2 classes.
Class: ['edible', 'poisonous']


In [5]:
converter = tf.lite.TFLiteConverter.from_saved_model(PRUNED_MODEL_PATH)

# Optimasi default
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Aktifkan FP16 (weight quantization)
converter.target_spec.supported_types = [tf.float16]

print("Converter FP16 siap")


Converter FP16 siap


In [6]:
print("🔄 Melakukan Post-Training Quantization FP16...")
tflite_fp16_model = converter.convert()

with open(TFLITE_FP16_PATH, "wb") as f:
    f.write(tflite_fp16_model)

model_size_mb = os.path.getsize(TFLITE_FP16_PATH) / (1024 * 1024)

print("Model FP16 berhasil disimpan")
print(f"Ukuran model FP16 : {model_size_mb:.2f} MB")


🔄 Melakukan Post-Training Quantization FP16...
Model FP16 berhasil disimpan
Ukuran model FP16 : 5.97 MB


In [7]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_FP16_PATH)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input dtype :", input_details[0]["dtype"])
print("Output dtype:", output_details[0]["dtype"])


Input dtype : <class 'numpy.float32'>
Output dtype: <class 'numpy.float32'>


In [56]:
y_true, y_pred = [], []

test_generator.reset()
start_time = time.time()

for i in range(len(test_generator)):
    image, label = test_generator[i]

    interpreter.set_tensor(
        input_details[0]["index"],
        image.astype(np.float32)
    )
    interpreter.invoke()

    output = interpreter.get_tensor(
        output_details[0]["index"]
    )

    y_true.append(np.argmax(label))
    y_pred.append(np.argmax(output))

end_time = time.time()


In [57]:
total_time = end_time - start_time
avg_time = total_time / len(test_generator)

print("\n=== HASIL PTQ FP16 ===")
print(f"Total inference time : {total_time:.4f} detik")
print(f"Rata-rata per image  : {avg_time:.6f} detik")

print("\nClassification Report FP16:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)



=== HASIL PTQ FP16 ===
Total inference time : 12.7334 detik
Rata-rata per image  : 0.054886 detik

Classification Report FP16:
              precision    recall  f1-score   support

      edible     0.9583    0.9583    0.9583       120
   poisonous     0.9554    0.9554    0.9554       112

    accuracy                         0.9569       232
   macro avg     0.9568    0.9568    0.9568       232
weighted avg     0.9569    0.9569    0.9569       232

